# 00 — Colab setup (run this first)

This notebook gets a fresh Google Colab session ready to train the five fundus
classifiers. Run the cells **top to bottom, once per session**.

It will:
1. confirm a GPU is attached,
2. mount Google Drive (so the dataset and the 3.7 GB RETFound weights persist
   across sessions and you don't re-download them every time),
3. install dependencies,
4. download the **Kaggle** fundus dataset (`linchundan/fundusimage1000`),
5. download the gated **RETFound** weights from Hugging Face,
6. rebuild the cached crops, manifest, and splits.

> **Before you start:** Runtime → Change runtime type → Hardware accelerator → **GPU**.

## 1. Confirm GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive (recommended)

Keeping the project on Drive means the dataset and weights survive a runtime
reset. Set `PROJECT_DIR` to where the repo should live on your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# where the project lives on your drive (edit to taste). this matches the
# /content/drive/MyDrive/... layout your manifest already uses.
PROJECT_DIR = "/content/drive/MyDrive/Drexel Biomedical AI/fundus_disease_classifier"

# set to your github repo. leave as-is if the folder already exists on drive.
REPO_URL = "https://github.com/acorn-lab/fundus_disease_classifier.git"

if not os.path.exists(PROJECT_DIR):
    os.makedirs(os.path.dirname(PROJECT_DIR), exist_ok=True)
    !git clone "$REPO_URL" "$PROJECT_DIR"

os.chdir(PROJECT_DIR)
print("working directory:", os.getcwd())
!ls

## 3. Install dependencies

Colab already ships torch + torchvision with CUDA, so this mainly adds `timm`,
`transformers`, `grad-cam`, and friends.

In [ ]:
!pip install -q -r requirements.txt

## 4. Download the Kaggle dataset

Source: **1000 Fundus images with 39 categories** (JSIEC, Cen et al. 2021) —
`kaggle.com/datasets/linchundan/fundusimage1000`.

You need a Kaggle API token. On kaggle.com: **Account → Settings → API → Create
New Token** downloads `kaggle.json`. Easiest in Colab: open the **🔑 Secrets**
panel (left sidebar) and add two secrets, `KAGGLE_USERNAME` and `KAGGLE_KEY`,
from that file. The cell below uses those secrets, or falls back to uploading
`kaggle.json` directly.

In [ ]:
# set up kaggle credentials (colab secrets first, else upload kaggle.json)
import os, json

os.makedirs("/root/.kaggle", exist_ok=True)
got_creds = False

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    got_creds = True
    print("using kaggle credentials from colab secrets")
except Exception as e:
    print("no colab secrets found, will ask you to upload kaggle.json:", e)

if not got_creds:
    from google.colab import files
    up = files.upload()  # choose your kaggle.json
    with open("/root/.kaggle/kaggle.json", "wb") as fh:
        fh.write(next(iter(up.values())))
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    print("kaggle.json saved")

In [ ]:
# download + unzip into ./1000images (skips if already present)
import os

if os.path.isdir("1000images") and len(os.listdir("1000images")) > 5:
    print("1000images/ already populated -- skipping download")
else:
    !pip install -q kaggle
    !kaggle datasets download -d linchundan/fundusimage1000 -p /content
    !mkdir -p 1000images
    !unzip -q -o /content/fundusimage1000.zip -d 1000images
    print("extracted to 1000images/")

# the kaggle zip nests a duplicate '1000images' folder; data.build_manifest
# already excludes it, so no cleanup is needed here.
!ls 1000images | head

## 5. Download the RETFound weights (gated)

The official RETFound weights live on a **gated** Hugging Face repo
(`YukunZhou/RETFound_mae_natureCFP`). One-time setup:

1. Make a free account at huggingface.co.
2. Open https://huggingface.co/YukunZhou/RETFound_mae_natureCFP and click
   **Agree / request access** (granted quickly).
3. Create a token: **Settings → Access Tokens → New token** (read scope).
4. Add it as a Colab secret named `HF_TOKEN`, or paste it when prompted below.

The cell saves the file as `RETFound_cfp_weights.pth` — the name the notebooks
expect.

In [ ]:
import os, shutil
from huggingface_hub import login, hf_hub_download, list_repo_files

OUT = "RETFound_cfp_weights.pth"

if os.path.exists(OUT) and os.path.getsize(OUT) > 1_000_000_000:
    print("RETFound weights already present -- skipping download")
else:
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
    login(token=token)  # opens a prompt if token is None

    repo_id = "YukunZhou/RETFound_mae_natureCFP"
    # find the .pth in the repo without hardcoding its exact name
    pth_files = [f for f in list_repo_files(repo_id) if f.endswith(".pth")]
    print("weight files in repo:", pth_files)
    src = hf_hub_download(repo_id=repo_id, filename=pth_files[0])
    shutil.copy(src, OUT)
    print("saved ->", OUT, f"({os.path.getsize(OUT) / 1e9:.2f} GB)")

## 6. Build cached crops, manifest, and splits

This applies the paper-style circular crop to every image and writes the
pixel-identical inputs every model reads from. Paths are environment-specific,
so `manifest_cached.json` is rebuilt here; `splits.json` is only rebuilt if
missing, to keep the frozen 5-fold assignment.

In [ ]:
import os, json
from fundus_lib import data, preprocess

# 1) manifest over the downloaded dataset (bigclasses 0-9)
manifest = data.build_manifest("1000images", keep_bigclasses=range(10))
print(f"{len(manifest)} images in bigclasses 0-9")

# 2) cache the cropped pngs (only if not already built)
if os.path.isdir("cache_crops") and len(os.listdir("cache_crops")) > 0:
    print("cache_crops/ exists -- rebuilding manifest paths against it")
manifest = preprocess.build_cache(manifest, "cache_crops", out_size=512)

# 3) persist the manifest with the new cache paths
with open("manifest_cached.json", "w") as fh:
    json.dump(manifest, fh)
print("wrote manifest_cached.json")

# 4) frozen splits -- keep existing file if present
if not os.path.exists("splits.json"):
    data.make_splits(manifest, n_splits=5, seed=42, out_path="splits.json")
    print("wrote splits.json")
else:
    print("splits.json already exists -- keeping the frozen folds")

## Done ✅

Your session is ready. Now open the model notebooks in order:
`01_cnn` → `02_resnet50` → `03_vit` → `04_dinov2` → `05_retfound`, then
`06_results` for the cross-model comparison.

Because everything lives on Drive, next session you can skip steps 4–6: the
dataset, weights, and crops will still be there.